In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Complete 5-Class ESI Multinomial Logistic Regressor with Class Weight Iterative Tuning (`models/lr_multiclass.ipynb`)

This notebook trains a **Complete 5-Class ESI Multinomial Logistic Regressor** with **Iterative Class Weight Tuning**:
- **Target Output**: Predicts all 5 emergency triage levels (**ESI 1, 2, 3, 4, 5**).
- **Feature Engineering Inputs**: Uses 13 clinical feature engineered inputs (age, gender, breathing difficulty, dyspnea flags, vital sign anomaly flags).
- **Strict Complete Case Filtering**: Removes any row with at least one NULL/NA feature across all splits.
- **Iterative Class Weight Tuning Grid**: Evaluates class weight multiplier configurations across sequential iterations to balance minority acuity levels (ESI 1 & ESI 5) against intermediate levels.
- **Visualization**: Plots Precision and Recall trajectories across tuning iterations for all 5 classes and saves chart to `plots/lr_multiclass_weight_tuning.png`.
- **Model Export**: Saves optimal tuned model to `deploy/lr_multiclass_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 Feature Engineered Inputs, & Apply Complete Case Analysis
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Construct 13 Clinical Feature Engineered Inputs
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))

initial_rows <- nrow(df_feng)
df <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df), nrow(df)))

cat(sprintf("Complete 5-Class ESI Dataset Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("5-Class ESI Distribution (Complete Cases):\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$raw_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize numeric features (center and scale)
numeric_cols <- names(train_df)[sapply(train_df, is.numeric)]
preproc <- preProcess(train_df[, numeric_cols], method = c("center", "scale"))

train_df[, numeric_cols] <- predict(preproc, train_df[, numeric_cols])
val_df[, numeric_cols]   <- predict(preproc, val_df[, numeric_cols])
test_df[, numeric_cols]  <- predict(preproc, test_df[, numeric_cols])

cat(sprintf("Complete Case Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Iterative Class Weight Tuning Loop over 5-Class Grid
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Define candidate class weight multipliers for ESI 1 and ESI 5 (relative to ESI 3 = 1.0, ESI 2 = 1.2, ESI 4 = 1.2)
w_grid <- expand.grid(
  w1 = c(1.0, 2.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0),
  w5 = c(1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0)
)
feat_names <- setdiff(names(train_df), "raw_esi")
formula_lr <- as.formula(paste("raw_esi ~", paste(feat_names, collapse = " + ")))

results_list <- list()
cat(sprintf("Starting 5-Class Class Weight Iterative Tuning over %d grid configurations...\n", nrow(w_grid)))

for (iter in 1:nrow(w_grid)) {
  w1_val <- w_grid$w1[iter]
  w5_val <- w_grid$w5[iter]
  
  weight_dict <- c("1" = w1_val, "2" = 1.2, "3" = 1.0, "4" = 1.2, "5" = w5_val)
  sample_weights <- as.numeric(weight_dict[as.character(train_df$raw_esi)])
  
  # Train 5-class model with current iteration class weights
  lr_iter <- multinom(formula_lr, data = train_df, weights = sample_weights, trace = FALSE, MaxNWts = 5000)
  
  # Evaluate on Validation set
  prob_val <- predict(lr_iter, newdata = val_df, type = "probs")
  max_idx  <- max.col(prob_val, ties.method = "first")
  pred_val <- factor(colnames(prob_val)[max_idx], levels = c("1", "2", "3", "4", "5"))
  act_val  <- factor(val_df$raw_esi, levels = c("1", "2", "3", "4", "5"))
  
  cm <- confusionMatrix(pred_val, act_val)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_cls <- cm$byClass[, "Pos Pred Value"]
  rec_by_cls  <- cm$byClass[, "Sensitivity"]
  
  macro_prec <- mean(prec_by_cls, na.rm = TRUE)
  macro_rec  <- mean(rec_by_cls,  na.rm = TRUE)
  macro_f1   <- 2 * (macro_prec * macro_rec) / (macro_prec + macro_rec + 1e-15)
  
  results_list[[iter]] <- data.frame(
    Iteration     = iter,
    Weight_ESI1   = w1_val,
    Weight_ESI5   = w5_val,
    Accuracy      = acc,
    Macro_Prec    = macro_prec,
    Macro_Rec     = macro_rec,
    Macro_F1      = macro_f1,
    Prec_ESI1     = ifelse(is.na(prec_by_cls["Class: 1"]), 0, prec_by_cls["Class: 1"]),
    Rec_ESI1      = ifelse(is.na(rec_by_cls["Class: 1"]), 0, rec_by_cls["Class: 1"]),
    Prec_ESI2     = ifelse(is.na(prec_by_cls["Class: 2"]), 0, prec_by_cls["Class: 2"]),
    Rec_ESI2      = ifelse(is.na(rec_by_cls["Class: 2"]), 0, rec_by_cls["Class: 2"]),
    Prec_ESI3     = ifelse(is.na(prec_by_cls["Class: 3"]), 0, prec_by_cls["Class: 3"]),
    Rec_ESI3      = ifelse(is.na(rec_by_cls["Class: 3"]), 0, rec_by_cls["Class: 3"]),
    Prec_ESI4     = ifelse(is.na(prec_by_cls["Class: 4"]), 0, prec_by_cls["Class: 4"]),
    Rec_ESI4      = ifelse(is.na(rec_by_cls["Class: 4"]), 0, rec_by_cls["Class: 4"]),
    Prec_ESI5     = ifelse(is.na(prec_by_cls["Class: 5"]), 0, prec_by_cls["Class: 5"]),
    Rec_ESI5      = ifelse(is.na(rec_by_cls["Class: 5"]), 0, rec_by_cls["Class: 5"])
  )
}
tuning_df <- do.call(rbind, results_list)
cat("5-Class Class Weight Iterative Tuning complete! Top 5 configurations by Validation Macro F1:\n")
print(head(tuning_df[order(-tuning_df$Macro_F1), c("Iteration", "Weight_ESI1", "Weight_ESI5", "Accuracy", "Macro_Prec", "Macro_Rec", "Macro_F1")], 5))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Plot Precision & Recall Trajectories Per Tuning Iteration
# ---------------------------------------------------------
tuning_long <- tuning_df %>%
  select(Iteration, Macro_Prec, Macro_Rec, Prec_ESI1, Rec_ESI1, Prec_ESI2, Rec_ESI2, Prec_ESI5, Rec_ESI5) %>%
  pivot_longer(cols = -Iteration, names_to = "Metric_Type", values_to = "Score")

p_tune <- ggplot(tuning_long, aes(x = Iteration, y = Score, color = Metric_Type)) +
  geom_line(size = 1.0) +
  geom_point(size = 1.8) +
  theme_minimal() +
  scale_color_manual(values = c(
    "Macro_Prec"   = "#2b5c8f",
    "Macro_Rec"    = "#e07a5f",
    "Prec_ESI1"    = "#d90429",
    "Rec_ESI1"     = "#ff4d6d",
    "Prec_ESI2"    = "#f77f00",
    "Rec_ESI2"     = "#fcbf49",
    "Prec_ESI5"    = "#1d3557",
    "Rec_ESI5"     = "#457b9d"
  )) +
  labs(
    title = "LR Multiclass (Complete 5-Class ESI): Precision & Recall Trajectories Per Tuning Iteration",
    subtitle = "Evaluating Macro and Per-Class Precision & Recall on Validation Set",
    x = "Tuning Iteration Number",
    y = "Metric Score"
  ) +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "right")

if (!dir.exists("../plots")) dir.create("../plots", recursive = TRUE)
ggsave("../plots/lr_multiclass_weight_tuning.png", plot = p_tune, width = 10, height = 5.5, dpi = 300)
cat("5-Class Class Weight Tuning Plot saved to: plots/lr_multiclass_weight_tuning.png\n")

p_tune

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Select Best Class Weights & Train Final 5-Class Model
# ---------------------------------------------------------
# Identify best iteration maximizing Validation Macro F1
best_row <- tuning_df[which.max(tuning_df$Macro_F1), ]
cat(sprintf("=== Selected Best Configuration (Iteration %d) ===\n", best_row$Iteration))
cat(sprintf("  Weight ESI 1 : %.1f\n", best_row$Weight_ESI1))
cat(sprintf("  Weight ESI 5 : %.1f\n", best_row$Weight_ESI5))
cat(sprintf("  Val Acc      : %.4f\n", best_row$Accuracy))
cat(sprintf("  Val Macro Prec: %.4f\n", best_row$Macro_Prec))
cat(sprintf("  Val Macro Rec : %.4f\n", best_row$Macro_Rec))
cat(sprintf("  Val Macro F1  : %.4f\n", best_row$Macro_F1))
best_weights_dict <- c("1" = best_row$Weight_ESI1, "2" = 1.2, "3" = 1.0, "4" = 1.2, "5" = best_row$Weight_ESI5)
final_weights <- as.numeric(best_weights_dict[as.character(train_df$raw_esi)])
cat("\nTraining Final 5-Class Model with Optimal Tuned Class Weights...\n")
lr_final <- multinom(formula_lr, data = train_df, weights = final_weights, trace = FALSE, MaxNWts = 5000)
# Function to compute PR-AUC
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_multiclass_lr <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "probs")
  target_classes <- c("1", "2", "3", "4", "5")
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$raw_esi, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  
  pr_auc_by_class <- numeric(5)
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(actual_factor == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- pROC::multiclass.roc(actual_factor, prob_matrix)
  macro_roc_auc <- as.numeric(roc_obj$auc)
  
  actual_table <- table(actual_factor)
  pred_table   <- table(pred_factor)
  
  diff_vec <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  class_comparison <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   OPTIMAL CLASS-WEIGHTED 5-CLASS LOGISTIC REGRESSOR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Target Class Count Comparison & Performance Summary Table:\n")
  print(class_comparison)
  
  cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}
# Benchmark Final Model on Validation set
evaluate_multiclass_lr(lr_final, val_df, "Validation")
# Benchmark Final Model on Test set
evaluate_multiclass_lr(lr_final, test_df, "Test")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Optimal 5-Class Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_multiclass_model.rds")
saveRDS(list(model = lr_final, preproc = preproc, best_weights = best_weights_dict), file = model_path)
cat("Optimal Class-Weighted 5-Class Logistic Regressor model saved to:", model_path, "\n")